## Setup

In [ ]:
import os
import h5py
import glob
import math

import pandas as pd
import numpy as np
import webdataset as wds

from utils.constant import *
from utils.paths import *

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

import timm
from timm.layers import SwiGLUPacked
from getpass import getpass
from huggingface_hub import login, hf_hub_download

from fsaa import attack

In [ ]:
login(getpass("Enter your HF token: "))

Enter your HF token: ··········


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
except ModuleNotFoundError:
    print("Not running on Google Colab. Skipping Drive mount.")

Mounted at /content/drive


In [5]:
def seed_torch(seed):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_torch(SEED)

## Dataset

In [7]:
def make_dataloader(patterns, labels_to_ids, split, batch_size, transform):
    def is_selected_class(sample):
        return sample[1]["label"] in labels_to_ids

    dataset = (
        wds.WebDataset(patterns[split], shardshuffle=False)
        .decode("pil")
        .to_tuple("jpg", "json", "__url__", '__key__')
        .select(is_selected_class)
        .map_tuple(
            transform,
            lambda x: labels_to_ids[x["label"]],
            lambda url: url,
            lambda key: key,
        )
        .map(lambda sample: (sample[0], sample[1], f"{os.path.basename(sample[2])}#{sample[3]}"))
    )
    return DataLoader(dataset, batch_size=batch_size)

## Pathology Foundation Models

In [11]:
def check_weights(model_name):
    weights_path = get_model_weights_path(model_name)

    if not os.path.exists(weights_path):
        repo_id = MODEL_TO_REPO[model_name]
        print(f"[DOWNLOADING] {model_name} weights from {repo_id}...")
        downloaded_path = hf_hub_download(repo_id=repo_id, filename='pytorch_model.bin', local_dir=os.path.dirname(weights_path))
        os.rename(downloaded_path, weights_path)
        print(f"Save to: {weights_path}")
    else:
        print(f"[INFO] {model_name} weights already downloaded to: {weights_path}")
    return weights_path

def get_standard_transforms(mean, std, img_size=224):
    base_transform = transforms.Compose([
        transforms.Resize(size=img_size, interpolation=transforms.InterpolationMode.BILINEAR, antialias=True),
        transforms.CenterCrop(size=(img_size, img_size)),
        transforms.ToTensor()
    ])
    normalize = transforms.Normalize(mean=mean, std=std)
    return base_transform, normalize

In [12]:
class Uni(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        self.model = timm.create_model("vit_large_patch16_224", img_size=224, patch_size=16, init_values=1e-5, num_classes=0, dynamic_img_size=True)
        self.model.load_state_dict(torch.load(weights_path, map_location="cpu"), strict=True)

    def forward(self, x):
        is_cuda = x.device.type == 'cuda'
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=is_cuda):
            output = self.model(x)
        return output

class Conch(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        from conch.open_clip_custom import create_model_from_pretrained
        self.model, _ = create_model_from_pretrained('conch_ViT-B-16', weights_path)

    def forward(self, x):
        image_embs = self.model.encode_image(x, proj_contrast=True, normalize=True)
        return image_embs

class Virchow(nn.Module):
    def __init__(self, weights_path):
        super().__init__()
        self.model = timm.create_model(
            "vit_huge_patch14_224",
            pretrained=False,
            img_size=224,
            init_values=1e-5,
            mlp_ratio=5.3375,
            mlp_layer=SwiGLUPacked,
            act_layer=torch.nn.SiLU,
            num_classes=0,
            global_pool='',
            dynamic_img_size=True,
        )
        self.model.load_state_dict(torch.load(weights_path, map_location="cpu"), strict=True)

    def forward(self, x):
        is_cuda = x.device.type == 'cuda'
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=is_cuda):
            output = self.model(x)
            class_token = output[:, 0]
            patch_tokens = output[:, 1:]
            embedding = torch.cat([class_token, patch_tokens.mean(1)], dim=-1)
        return embedding

In [13]:
def get_model_and_transforms(model_name):
    """Load the Pathology Foundation Model and return the model, embedding size, base transform, and normalization transform."""
    check_supported_models('model_name', model_name)
    if model_name == 'uni_v1':
        weights_path = check_weights('uni_v1')
        model = Uni(weights_path)
        base_transform, normalize = get_standard_transforms(IMAGENET_MEAN, IMAGENET_STD)
        embed_dim = MODEL2EMB_DIM[model_name]
        return model, embed_dim, base_transform, normalize
    elif model_name == 'conch_v1':
        weights_path = check_weights('conch_v1')
        model = Conch(weights_path)
        base_transform, normalize = get_standard_transforms(OPENAI_MEAN, OPENAI_STD)
        embed_dim = MODEL2EMB_DIM[model_name]
        return model, embed_dim, base_transform, normalize
    elif model_name == 'virchow_v1':
        weights_path = check_weights('virchow_v1')
        model = Virchow(weights_path)
        base_transform, normalize = get_standard_transforms(IMAGENET_MEAN, IMAGENET_STD)
        embed_dim = MODEL2EMB_DIM[model_name]
        return model, embed_dim, base_transform, normalize
    else:
        raise ValueError(f"Model '{model_name}' not supported.")

## Feature Extraction from Clean Images with and without Gaussian Smoothing Defense

In [14]:
def extract_clean_features(model_name, model, feature_dim, dataloader, save_path, split, sigma=None):
    """Extract features from clean images and store them in an HDF5 file."""

    tag = f'CLEAN+DEF (sigma={sigma})' if sigma is not None else 'CLEAN'
    print(f"[{tag} FEATURE EXTRACTION] Processing {split} images using {model_name}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    with h5py.File(save_path, 'w') as hdf5_file:
        features_dset = hdf5_file.create_dataset(
            'features', shape=(0, feature_dim), maxshape=(None, feature_dim), chunks=True, dtype="float32"
        )
        labels_dset = hdf5_file.create_dataset(
            'labels', shape=(0,), maxshape=(None,), chunks=True, dtype="int64"
        )
        sample_names_dset = hdf5_file.create_dataset(
            "sample_names", shape=(0,), maxshape=(None,), chunks=True, dtype=h5py.string_dtype(encoding="utf-8"),
        )

        with torch.inference_mode():
            for images, labels, sample_names in dataloader:
                images = images.to(device, non_blocking=True)

                features = model(images).cpu().numpy().astype("float32")
                labels = labels.numpy()

                curr_batch_size = features.shape[0]

                features_dset.resize(features_dset.shape[0] + curr_batch_size, axis=0)
                features_dset[-curr_batch_size:] = features

                labels_dset.resize(labels_dset.shape[0] + curr_batch_size, axis=0)
                labels_dset[-curr_batch_size:] = labels

                sample_names_dset.resize(sample_names_dset.shape[0] + curr_batch_size, axis=0)
                sample_names_dset[-curr_batch_size:] = sample_names
    print(f"Saved to: {save_path}")

def run_clean_feature_extraction(model_name, patterns=PATTERNS, labels_to_ids=LABELS_TO_IDS, splits=SPLITS, batch_size=128, sigma=None):
    """
    Run feature extraction with the specified model on clean images for the specified dataset splits.
    Optionally apply Gaussian smoothing with the specified sigma value before feature extraction.
    Skip splits for which features have already been saved.
    """
    for split in splits:
        tag = f'CLEAN+DEF (sigma={sigma})' if sigma is not None else 'CLEAN'
        print(f"\n[{tag} FEATURE EXTRACTION] Model name: {model_name}")
        print(f"[{tag} FEATURE EXTRACTION] Split: {split}")
        clean_feat_path = get_clean_features_path(model_name, split, sigma)
        if os.path.exists(clean_feat_path):
            print(f"[{tag} FEATURE EXTRACTION] Features already extracted and saved to {clean_feat_path}, skipping.")
            continue
        model, feature_dim, base_transform, normalize = get_model_and_transforms(model_name)
        if sigma is not None:
            kernel_size = 2 * math.ceil(3 * sigma) + 1
            gaussian_filter = transforms.GaussianBlur(kernel_size=kernel_size, sigma=sigma)
            transform = transforms.Compose([base_transform, gaussian_filter, normalize])
        else:
            transform = transforms.Compose([base_transform, normalize])
        loader = make_dataloader(patterns=patterns, labels_to_ids=labels_to_ids, split=split, batch_size=batch_size, transform=transform)
        extract_clean_features(model_name=model_name, model=model, feature_dim=feature_dim, dataloader=loader, save_path=clean_feat_path, split=split, sigma=sigma)

In [15]:
for model_name in SUPPORTED_MODELS:
    run_clean_feature_extraction(model_name)


[CLEAN FEATURE EXTRACTION] Model name: uni_v1
[CLEAN FEATURE EXTRACTION] Split: train
[CLEAN FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/uni_v1/train.h5, skipping.

[CLEAN FEATURE EXTRACTION] Model name: uni_v1
[CLEAN FEATURE EXTRACTION] Split: valid
[CLEAN FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/uni_v1/valid.h5, skipping.

[CLEAN FEATURE EXTRACTION] Model name: uni_v1
[CLEAN FEATURE EXTRACTION] Split: test
[CLEAN FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/uni_v1/test.h5, skipping.

[CLEAN FEATURE EXTRACTION] Model name: conch_v1
[CLEAN FEATURE EXTRACTION] Split: train
[CLEAN FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/conch_v1/train.h5, skipping.

[CLEAN FEATURE EXTRACTION] Model name: conch_v1
[CLEAN FEATURE EXTRACTION] Split:

In [16]:
for model_name in SUPPORTED_MODELS:
    for sigma in SIGMA:
        run_clean_feature_extraction(model_name=model_name, splits=['test'], sigma=sigma)


[CLEAN+DEF (sigma=0.5) FEATURE EXTRACTION] Model name: uni_v1
[CLEAN+DEF (sigma=0.5) FEATURE EXTRACTION] Split: test
[CLEAN+DEF (sigma=0.5) FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/uni_v1/test_sigma0.5.h5, skipping.

[CLEAN+DEF (sigma=1) FEATURE EXTRACTION] Model name: uni_v1
[CLEAN+DEF (sigma=1) FEATURE EXTRACTION] Split: test
[CLEAN+DEF (sigma=1) FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/uni_v1/test_sigma1.h5, skipping.

[CLEAN+DEF (sigma=1.5) FEATURE EXTRACTION] Model name: uni_v1
[CLEAN+DEF (sigma=1.5) FEATURE EXTRACTION] Split: test
[CLEAN+DEF (sigma=1.5) FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/clean/uni_v1/test_sigma1.5.h5, skipping.

[CLEAN+DEF (sigma=2) FEATURE EXTRACTION] Model name: uni_v1
[CLEAN+DEF (sigma=2) FEATURE EXTRACTION] Split: test
[CLEAN+DEF (sigma=2) FEATURE EXTRACTION] F

## Feature Space Adversarial Attack and Generation of Adversarial Images

In [17]:
def get_adv_images(model_name, model, normalize, dataloader, split, psnr):
    """Generate and save adversarial images as PyTorch tensor files"""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.eval().to(device)
    max_img_mse = MSE_TARGETS[psnr]
    for batch_idx, (images, labels, sample_names) in enumerate(dataloader):
        save_path = get_adv_images_path(model_name, psnr, split, batch_idx)
        if os.path.exists(save_path):
            print(f"[ADVERSARIAL IMAGE GENERATION] Batch {batch_idx} already processed and saved to {save_path}, skipping.")
            continue

        images = images.to(device)
        adv_images = attack(
            model=model,
            images=images,
            transform=normalize,
            steps= 350,
            optimizer_kwargs={'lr': 5e-4},
            feature_loss=torch.nn.CosineSimilarity(dim=-1),
            flw=1.0,
            initial_noise_scale=math.sqrt(max_img_mse) * 0.3,
            max_img_mse=max_img_mse,
            pbar=True
        )

        tmp_path = save_path + ".tmp"
        torch.save({
            "adv_images": adv_images.detach().cpu(),
            "labels": labels,
            "sample_names": sample_names
        }, tmp_path)
        os.replace(tmp_path, save_path)
        print(f"[ADVERSARIAL IMAGES GENERATION] Batch {batch_idx} saved to: {save_path}")

def run_fsaa(model_name, psnr, splits, batch_size):
    """
    Run the Feature Space Adversarial Attack to generate adversarial images with the specified PSNR target,
    using the specified model for the specified dataset splits.
    """
    model, feature_dim, base_transform, normalize = get_model_and_transforms(model_name)
    for split in splits:
        loader = make_dataloader(patterns=PATTERNS, labels_to_ids=LABELS_TO_IDS, split=split, batch_size=batch_size, transform=base_transform)
        get_adv_images(model_name=model_name, model=model, normalize=normalize, dataloader=loader, split=split, psnr=psnr)

In [18]:
for psnr, mse in MSE_TARGETS.items():
    run_fsaa(model_name='uni_v1', psnr=psnr, splits=['test'], batch_size=16)

[INFO] uni_v1 weights already downloaded to: /content/drive/MyDrive/fsaa_pfms/model_weights/uni_v1.bin
[ADVERSARIAL IMAGE GENERATION] Batch 0 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/uni_v1/psnr_45dB/test_batch_0000.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 1 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/uni_v1/psnr_45dB/test_batch_0001.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 2 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/uni_v1/psnr_45dB/test_batch_0002.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 3 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/uni_v1/psnr_45dB/test_batch_0003.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 4 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/uni_v1/psnr_45dB/test_batch_0004.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 5 already processed and saved to /content/drive/MyDrive

In [19]:
for psnr, mse in MSE_TARGETS.items():
    run_fsaa(model_name='conch_v1', psnr=psnr, splits=['test'], batch_size=16)

[INFO] conch_v1 weights already downloaded to: /content/drive/MyDrive/fsaa_pfms/model_weights/conch_v1.bin
[ADVERSARIAL IMAGE GENERATION] Batch 0 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/conch_v1/psnr_45dB/test_batch_0000.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 1 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/conch_v1/psnr_45dB/test_batch_0001.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 2 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/conch_v1/psnr_45dB/test_batch_0002.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 3 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/conch_v1/psnr_45dB/test_batch_0003.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 4 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/conch_v1/psnr_45dB/test_batch_0004.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 5 already processed and saved to /content

In [20]:
for psnr, mse in MSE_TARGETS.items():
    run_fsaa(model_name='virchow_v1', psnr=psnr, splits=['test'], batch_size=16)

[INFO] virchow_v1 weights already downloaded to: /content/drive/MyDrive/fsaa_pfms/model_weights/virchow_v1.bin
[ADVERSARIAL IMAGE GENERATION] Batch 0 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/virchow_v1/psnr_45dB/test_batch_0000.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 1 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/virchow_v1/psnr_45dB/test_batch_0001.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 2 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/virchow_v1/psnr_45dB/test_batch_0002.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 3 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/virchow_v1/psnr_45dB/test_batch_0003.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 4 already processed and saved to /content/drive/MyDrive/fsaa_pfms/adv_images/virchow_v1/psnr_45dB/test_batch_0004.pt, skipping.
[ADVERSARIAL IMAGE GENERATION] Batch 5 already processed and sav

## Feature Extraction from Adversarial Images with and without Gaussian Smoothing Defense

In [21]:
def extract_adv_features(model_name, model, transform, feature_dim, adv_imgs_dir, save_path, split, sigma=None):
    """Extract features from adversarial images and store them in an HDF5 file."""
    tag = f"ADV+DEF (sigma={sigma})" if sigma is not None else "ADVERSARIAL"
    print(f"[{tag} FEATURE EXTRACTION] Processing {split} images using {model_name}...")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval().to(device)

    adv_files = sorted(glob.glob(os.path.join(adv_imgs_dir, f"{split}_batch_*.pt")))
    with h5py.File(save_path, 'w') as hdf5_file:
        features_dset = hdf5_file.create_dataset(
            'features', shape=(0, feature_dim), maxshape=(None, feature_dim), chunks=True, dtype="float32"
        )
        labels_dset = hdf5_file.create_dataset(
            'labels', shape=(0,), maxshape=(None,), chunks=True, dtype="int64"
        )
        sample_names_dset = hdf5_file.create_dataset(
            "sample_names", shape=(0,), maxshape=(None,), chunks=True, dtype=h5py.string_dtype(encoding="utf-8")
        )

        with torch.inference_mode():
            for adv_file in adv_files:
                data = torch.load(adv_file)
                adv_images = data["adv_images"].to(device, non_blocking=True)
                features = model(transform(adv_images)).flatten(1).cpu().numpy().astype("float32")

                labels = data["labels"]
                if isinstance(labels, torch.Tensor):
                    labels = labels.numpy().astype("int64")
                sample_names = data["sample_names"]
                curr_batch_size = features.shape[0]

                features_dset.resize(features_dset.shape[0] + curr_batch_size, axis=0)
                features_dset[-curr_batch_size:] = features

                labels_dset.resize(labels_dset.shape[0] + curr_batch_size, axis=0)
                labels_dset[-curr_batch_size:] = labels

                sample_names_dset.resize(sample_names_dset.shape[0] + curr_batch_size, axis=0)
                sample_names_dset[-curr_batch_size:] = sample_names
    print(f"Saved to: {save_path}")

def run_adv_feature_extraction(target_model_name, source_model_name, psnr, splits=SPLITS, sigma=None):
    """
    Run feature extraction with the target model on adversarial images generated using the source model
    at the specified PSNR target.
    Optionally apply Gaussian smoothing with the specified sigma value before feature extraction.
    Skip splits for which features have already been saved.
    """
    for split in splits:
        tag = f"ADV+DEF sigma{sigma}" if sigma is not None else "ADVERSARIAL"
        print(f"\n[{tag} FEATURE EXTRACTION] Target model name: {target_model_name}")
        print(f"[{tag} FEATURE EXTRACTION] Source model name: {source_model_name}")
        print(f"[{tag} FEATURE EXTRACTION] PSNR: {psnr}")
        print(f"[{tag} FEATURE EXTRACTION] Split: {split}")
        adv_feat_path = get_adv_features_path(target_model_name, source_model_name, split, psnr, sigma)
        if os.path.exists(adv_feat_path):
            print(f"[{tag} FEATURE EXTRACTION] Features already extracted and saved to {adv_feat_path}, skipping.")
            continue
        adv_imgs_dir = os.path.dirname(get_adv_images_path(source_model_name, MSE_TARGETS[psnr], split, batch_idx=0))
        model, feature_dim, base_transform, normalize = get_model_and_transforms(target_model_name)
        if sigma is not None:
            kernel_size = 2 * math.ceil(3 * sigma) + 1
            gaussian_filter = transforms.GaussianBlur(kernel_size=kernel_size, sigma=sigma)
            transform = transforms.Compose([gaussian_filter, normalize])
        else:
            transform = normalize
        extract_adv_features(target_model_name, model, transform, feature_dim, adv_imgs_dir, save_path=adv_feat_path, split=split, sigma=sigma)

In [22]:
for psnr, mse in MSE_TARGETS.items():
    for source_m in SUPPORTED_MODELS:
        for target_m in SUPPORTED_MODELS:
            run_adv_feature_extraction(target_m, source_m, splits=['test'], psnr=psnr)


[ADVERSARIAL FEATURE EXTRACTION] Target model name: uni_v1
[ADVERSARIAL FEATURE EXTRACTION] Source model name: uni_v1
[ADVERSARIAL FEATURE EXTRACTION] PSNR: 45dB
[ADVERSARIAL FEATURE EXTRACTION] Split: test
[ADVERSARIAL FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/adversarial/uni_v1/psnr_45dB/test_wb.h5, skipping.

[ADVERSARIAL FEATURE EXTRACTION] Target model name: conch_v1
[ADVERSARIAL FEATURE EXTRACTION] Source model name: uni_v1
[ADVERSARIAL FEATURE EXTRACTION] PSNR: 45dB
[ADVERSARIAL FEATURE EXTRACTION] Split: test
[ADVERSARIAL FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/adversarial/conch_v1/psnr_45dB/test_transf_uni_v1.h5, skipping.

[ADVERSARIAL FEATURE EXTRACTION] Target model name: virchow_v1
[ADVERSARIAL FEATURE EXTRACTION] Source model name: uni_v1
[ADVERSARIAL FEATURE EXTRACTION] PSNR: 45dB
[ADVERSARIAL FEATURE EXTRACTION] Split: test
[ADVERSARIAL FEATURE EXTR

In [23]:
for model_name in SUPPORTED_MODELS:
    for psnr, mse in MSE_TARGETS.items():
        for sigma in SIGMA:
            run_adv_feature_extraction(model_name, model_name, splits=['test'], psnr=psnr, sigma=sigma)


[ADV+DEF sigma0.5 FEATURE EXTRACTION] Target model name: uni_v1
[ADV+DEF sigma0.5 FEATURE EXTRACTION] Source model name: uni_v1
[ADV+DEF sigma0.5 FEATURE EXTRACTION] PSNR: 45dB
[ADV+DEF sigma0.5 FEATURE EXTRACTION] Split: test
[ADV+DEF sigma0.5 FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/adversarial/uni_v1/psnr_45dB/test_wb_sigma0.5.h5, skipping.

[ADV+DEF sigma1 FEATURE EXTRACTION] Target model name: uni_v1
[ADV+DEF sigma1 FEATURE EXTRACTION] Source model name: uni_v1
[ADV+DEF sigma1 FEATURE EXTRACTION] PSNR: 45dB
[ADV+DEF sigma1 FEATURE EXTRACTION] Split: test
[ADV+DEF sigma1 FEATURE EXTRACTION] Features already extracted and saved to /content/drive/MyDrive/fsaa_pfms/features/adversarial/uni_v1/psnr_45dB/test_wb_sigma1.h5, skipping.

[ADV+DEF sigma1.5 FEATURE EXTRACTION] Target model name: uni_v1
[ADV+DEF sigma1.5 FEATURE EXTRACTION] Source model name: uni_v1
[ADV+DEF sigma1.5 FEATURE EXTRACTION] PSNR: 45dB
[ADV+DEF sigma1.5 